# Paired leave-one-pair-out jackknife for every delta-KID; one-session rescoring; trace-corrected KID

The final scoring pass behind `tab:eqsample`, `tab:generative`/`tab:app-generative`, and every delta-KID confidence interval in the paper. Cell 5 prints (but does not save to any file) every trace-corrected absolute-KID macro (`\GKID*`, `\TMKID*`, `\EqKID*`, `\GEKID*`, `\TMkKID*`).

**Needs:** the cached feature files from `4_1_*` and `4_2_*`.

**Maintained runnable path:** `score_only.py` recomputes every one of these values (with the trace-corrected reference term) directly from the archive's cached features, no rendering needed. This notebook is the reference for exactly how that correction was derived.

# Paired jackknife confidence intervals for every ΔKID claim

The paper's ΔKID intervals were computed by m-of-n subsampling (100 of 120 pairs, without replacement), which understates uncertainty. This notebook recomputes each interval by a **paired leave-one-pair-out jackknife** on the same images, with the canonical reference and extractor, and prints the `\newcommand` lines to paste into the numbers file.

Cell 1 re-extracts Inception features from the saved PNGs (Texture Mixer coupling ablation, equal-sample CNT ablation) and loads the cached features of the Gaussian-primitive study; cell 2 computes the intervals. About 5 minutes after setup.


In [ ]:
# ============================================================
# 0. SETUP — fast canonical setup (DTD from Drive zip, detectron2 only if the import fails) — canonical protocol (finish_day / mechanism cell 0, verbatim) + pinned CNT commit
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile,hashlib,itertools
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
assert torch.cuda.is_available(),'switch runtime to GPU'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'
MECH=f'{BASE}/mechanism'; os.makedirs(MECH,exist_ok=True); FIGS=f'{MECH}/figs'; os.makedirs(FIGS,exist_ok=True)
TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
t0=time.time()

# --- DTD ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system('rm -rf /content/_dtd'); os.system(f'unzip -qo "{ZIP}" -d /content/_dtd')
    inner=os.listdir('/content/_dtd'); src='/content/_dtd/images' if 'images' in inner else f'/content/_dtd/{inner[0]}'
    os.system(f'rm -rf {LOCAL} && mv "{src}" {LOCAL}')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 && cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
CNT_COMMIT=os.popen(f'cd {CNT} && git rev-parse HEAD').read().strip()
print(f'model ready {time.time()-t0:.0f}s | FMS={FMS} | CNT commit {CNT_COMMIT[:10]}')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def _pyr(si):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(FMS): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return sp
def build_style_maps(bf,si,fms):
    sp=_pyr(si); return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def decode_stagewise(blob_ref,sm_by_stage):
    """same decoder, but stage i receives sm_by_stage[i] (dict size->map). For E5 per-stage swaps."""
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm_by_stage[i][fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def sm_from_field(Z,L,H,W,Cc):
    it=Z.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
def _dec(d,Z,L,H,W,Cc,layout='a'): return decode_with_style_maps(d["blob_"+layout],sm_from_field(Z.float(),L,H,W,Cc))
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def pair_imgs(k): return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

# --- canonical Sinkhorn (as used in the paper) + converged log-domain Sinkhorn with duals ---
def sinkhorn_plan(x,y,eps,iters=ITERS):
    Cm=torch.cdist(x,y).pow(2); K=torch.exp(-Cm/eps)
    u=torch.ones(len(x),device=x.device)/len(x); v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u); b=torch.ones_like(v)
    for _ in range(iters): a=u/(K@b+1e-30); b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:], Cm
def sinkhorn_log(x,y,eps,iters=3000,tol=1e-6):
    """balanced, uniform marginals, float64 log-domain; returns P, duals f,g, cost, marginal residual, iterations."""
    X=x.double(); Y=y.double(); Cm=torch.cdist(X,Y).pow(2); n,m=Cm.shape
    la=-math.log(n); lb=-math.log(m); f=torch.zeros(n,device=X.device,dtype=torch.float64); g=torch.zeros(m,device=X.device,dtype=torch.float64)
    for it in range(1,iters+1):
        f=eps*(la-torch.logsumexp((g[None,:]-Cm)/eps,1)); g=eps*(lb-torch.logsumexp((f[:,None]-Cm)/eps,0))
        if it%10==0:
            logP=(f[:,None]+g[None,:]-Cm)/eps; res=float((torch.exp(logP).sum(1)-1/n).abs().max()*n)
            if res<tol: break
    logP=(f[:,None]+g[None,:]-Cm)/eps; P=torch.exp(logP)
    res=float(max((P.sum(1)-1/n).abs().max()*n,(P.sum(0)-1/m).abs().max()*m))
    return P,f,g,Cm,res,it
def plan_stats(P):
    P=P.double(); n=P.shape[0]; q=P/(P.sum(1,keepdim=True)+1e-300)
    Hrow=float(-(q*torch.log(q+1e-300)).sum(1).mean()); Htot=float(-(P*torch.log(P+1e-300)).sum())
    return dict(H_row=Hrow,H_total=Htot,MI_nats=math.log(n)-Hrow)
def natural_extension(x_all,y,g,eps,chunk=1024):
    """T(x)=sum_j q_j(x) y_j, q_j ∝ exp((g_j-||x-y_j||²)/eps) — dual-potential extension of a converged plan."""
    out=[]
    for i in range(0,len(x_all),chunk):
        Cm=torch.cdist(x_all[i:i+chunk].double(),y.double()).pow(2); q=torch.softmax((g[None,:]-Cm)/eps,1); out.append(q@y.double())
    return torch.cat(out)

# --- matching rules -> sampled targets (M,d); full field via canonical nn extension ---
def _targets(sf,tf,rule,seed,eps=EPS,M=NSUB,n_exact=1024,plan_kind='canonical'):
    seed=int(seed)   # numpy int64 from pandas breaks torch manual_seed
    N=sf.shape[0]; i_s=_sub(N,seed,'s')[:M]; i_t=_sub(N,seed,'t')[:M]; src,tgt=sf[i_s],tf[i_t]; info={}
    if rule in('ot','ot_sharp','ot_soft'):
        e={'ot':eps,'ot_sharp':0.005,'ot_soft':0.5}[rule] if rule!='ot' else eps
        if plan_kind=='canonical': P,_=sinkhorn_plan(src,tgt,e); info=dict(eps=e,**plan_stats(P))
        else: P,f,g,_,res,it=sinkhorn_log(src,tgt,e); info=dict(eps=e,res=res,iters=it,**plan_stats(P)); info['g']=g
        tt=((P/(P.sum(1,keepdim=True)+1e-30))@tgt.double()).float() if P.dtype==torch.float64 else (P/(P.sum(1,keepdim=True)+1e-30))@tgt
    elif rule=='exact':
        g_=torch.Generator(device=device).manual_seed(seed+11)
        js=torch.randperm(M,generator=g_,device=device)[:n_exact]; is_=torch.randperm(M,generator=g_,device=device)[:n_exact]
        r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
        tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
        ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]; info=dict(n_exact=n_exact)
    elif rule=='nn': tt=tgt[torch.cdist(src,tgt).argmin(1)]
    elif rule=='gauss':
        X,Y=src.double(),tgt.double(); mx,my=X.mean(0),Y.mean(0); Xc=X-mx; Yc=Y-my; Cc=X.shape[1]; I=torch.eye(Cc,device=device,dtype=torch.float64); ridge=1e-4
        Sa=(Xc.T@Xc)/(len(X)-1)+ridge*I; Sb=(Yc.T@Yc)/(len(Y)-1)+ridge*I
        def ps(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.sqrt())@V.T
        def pis(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.rsqrt())@V.T
        Sh=ps(Sa,ridge); Sih=pis(Sa,ridge); Amap=Sih@ps(Sh@Sb@Sh,0.0)@Sih; Amap=.5*(Amap+Amap.T)
        tt=(my+Xc@Amap.T).float(); info=dict(A=Amap,mu=(mx,my))
    elif rule=='uniform': tt=tgt.mean(0,keepdim=True).expand_as(src).clone()
    elif rule=='random':
        g_=torch.Generator(device=device).manual_seed(seed+7); tt=tgt[torch.randperm(M,generator=g_,device=device)]
    elif rule=='meanshift': tt=src+(tgt.mean(0)-src.mean(0))      # CNT-style native mean shift, adapted to the interpolation contract
    else: raise ValueError(rule)
    return i_s,tt,info
def field(d,rule,seed,**kw):
    seed=int(seed)
    """full-grid target field U (N,d) with the canonical nn extension; also returns X,Y grids and plan info."""
    L,Cc,H,W,sf,tf=_grids(d); i_s,tt,info=_targets(sf,tf,rule,seed,**kw)
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return dict(U=tt[nn],X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,i_s=i_s,tt=tt,nn=nn,info=info)
def zt(fd,t): return (1-t)*fd['X']+t*fd['U']
def decode_field(d,fd,Z): return _dec(d,Z,fd['L'],fd['H'],fd['W'],fd['Cc'])

# --- fixed-layout subspace of A: X = W_A F_A ---
def splat_W(d,side='a'):
    sp=_pyr(d['si_'+side]); Wm=sp[max(FMS)][0]; K=Wm.shape[0]
    return Wm.reshape(K,-1).T.contiguous().double(), d['bf_'+side][0].double()      # (N,K),(K,d)
def projector(Wm,tol=1e-6):
    Uw,S,_=torch.linalg.svd(Wm,full_matrices=False); keep=S>S[0]*tol; return Uw[:,keep], int(keep.sum())
def proj(Q,U): U=U.double(); return Q@(Q.T@U)
def e_sub(Q,U):
    U=U.double(); Uc=U-U.mean(0,keepdim=True); off=U-proj(Q,U)
    return dict(e_sub=float(off.norm()/(Uc.norm()+1e-12)),off_rms=float(off.norm()/math.sqrt(U.numel())),cen_rms=float(Uc.norm()/math.sqrt(U.numel())))
def e_aff(X,U,seed=0):
    """affine fit U≈XA+b on half the rows, residual on held-out rows."""
    X=X.double(); U=U.double(); N=len(X); g=torch.Generator(device=X.device).manual_seed(seed); p=torch.randperm(N,generator=g,device=X.device); tr,te=p[:N//2],p[N//2:]
    Xa=torch.cat([X,torch.ones(N,1,device=X.device,dtype=torch.float64)],1)
    G=Xa[tr].T@Xa[tr]; sol=torch.linalg.solve(G+1e-6*torch.trace(G)/G.shape[0]*torch.eye(G.shape[0],device=G.device,dtype=G.dtype),Xa[tr].T@U[tr]); R=U[te]-Xa[te]@sol; Uc=U[te]-U[te].mean(0,keepdim=True)
    return dict(e_aff=float(R.norm()/(Uc.norm()+1e-12)),aff_rms=float(R.norm()/math.sqrt(R.numel())))
def commutation_err(Q,Wm,Fm,Tfun):
    """J = T(W F) - W T(F), for a frozen map T applied to blob vectors (out-of-support warning if far from cloud)."""
    X=(Wm@Fm); TX=Tfun(X.float()).double(); TF=Tfun(Fm.float()).double(); J=TX-Wm@TF
    return dict(comm=float(J.norm()/(TX-TX.mean(0)).norm()),comm_rms=float(J.norm()/math.sqrt(J.numel())))
def neighbor_disc(U,H,W):
    G=U.reshape(H,W,-1); return float(((G[:,1:]-G[:,:-1]).norm(dim=-1).mean()+(G[1:]-G[:-1]).norm(dim=-1).mean())/2)
def var_ret(U,Y): return float(U.double().var(0).sum()/Y.double().var(0).sum())
def interaction_scale(X,Y):
    """RMS of -2 Xc Yc^T (centered interaction) vs raw median cost: the calibration the plan asks for."""
    Xc=X.double()-X.double().mean(0); Yc=Y.double()-Y.double().mean(0); M_=-2*Xc@Yc.T
    return dict(inter_rms=float(M_.pow(2).mean().sqrt()),cost_med=float(torch.cdist(X.double(),Y.double()).pow(2).median()))

# --- observers (canonical) ---
import timm, lpips as lpips_lib
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
lp=lpips_lib.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p:(torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
SCORE_RES=128
def at128(pil): return pil.convert('RGB').resize((SCORE_RES,SCORE_RES))
ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def kid_point(X,Y,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(len(X),len(Y)); x=X[:m]; y=Y[rng.choice(len(Y),m,replace=False)]
    kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
KS120=[int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k]; KSET=set(KS120)
DEV=[int(p_['k']) for p_ in MAN_ALL if int(p_['k']) not in KSET][:24]          # development set: pair-disjoint from the 120 confirmation pairs
print('dev pairs:',DEV); print('confirmation pairs:',len(KS120))

# --- per-pair scoring bundle for a decoded midpoint (human-study proxies) ---
_anc={}
def anchors(k):
    if k in _anc: return _anc[k]
    A,B=pair_imgs(k); rA=to_pil(make_recon(A)); rB=to_pil(make_recon(B)); f=dino_feats([at128(rA),at128(rB)])
    _anc[k]=dict(fA=f[0],fB=f[1],dAB=float(np.linalg.norm(f[0]-f[1])),rA=rA,rB=rB); return _anc[k]
def reencode_field(pil):
    e=cnt_load(pil); ba,bfa,sia=get_gen_inputs(e); sm=build_style_maps(bfa,sia,FMS); L=max(FMS); A=sm[L][0]; Cc=A.shape[0]
    return A.permute(1,2,0).reshape(-1,Cc)
def score_image(k,pil,Z=None,t=None):
    a=anchors(k); im=at128(pil); fi=inc_feats([im])[0]; fd=dino_feats([im])[0]
    r=dict(real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]),
           dA=float(np.linalg.norm(fd-a['fA'])/a['dAB']),dB=float(np.linalg.norm(fd-a['fB'])/a['dAB']))
    r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    if Z is not None:
        Xr=reencode_field(pil); r['roundtrip']=float((Xr.double()-Z.double()).norm()/(Z.double().norm()+1e-12))
    return r,fi,fd
_cache_d={}
def get_d(k):
    if k not in _cache_d:
        if len(_cache_d)>64: _cache_d.clear()
        A,B=pair_imgs(k); _cache_d[k]=build_d(cnt_load(A),cnt_load(B))
    return _cache_d[k]
def save_rows(rows,path): pd.DataFrame(rows).to_csv(path,index=False)
def load_rows(path,keys):
    if os.path.exists(path):
        df=pd.read_csv(path); return df.to_dict('records'), set(tuple(r[c] for c in keys) for r in df.to_dict('records'))
    return [],set()
print(f'setup done in {time.time()-t0:.0f}s')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False; torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
print('precision: fp32, deterministic cuDNN')


In [ ]:
# ============================================================
# 1. FEATURES — re-extract Inception features from the saved midpoints (canonical at128 + inc_feats) and cache them
# ============================================================
SCALE=f'{BASE}/tm_scale1000'; JK=f'{BASE}/jackknife'; os.makedirs(JK,exist_ok=True)
KS120=sorted(int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k); KS200=[int(p_['k']) for p_ in MAN_ALL[:200]]
def feats_dir(d,ks,name):
    fp=f'{JK}/{name}.npy'
    if os.path.exists(fp): return np.load(fp)
    missing=[k for k in ks if not os.path.exists(f'{d}/{k:04d}.png')]; assert not missing,f'{d}: missing {missing[:5]}'
    X=inc_feats([at128(Image.open(f'{d}/{k:04d}.png').convert('RGB')) for k in ks]).astype(np.float64); np.save(fp,X); return X
t0=time.time()
TM={r:feats_dir(f'{SCALE}/tmc/{r}',KS120,f'tm_{r}') for r in ['lerp','mean','ot_r0.03','hard']}; print('TM features',{r:X.shape for r,X in TM.items()},f'{time.time()-t0:.0f}s')
EQ={r:feats_dir(f'{OUT}/eqsample/{r}_t0.5',KS200,f'eq_{r}') for r in ['hard','ot_e0.005','ot_e0.05']}; print('equal-sample features',{r:X.shape for r,X in EQ.items()},f'{time.time()-t0:.0f}s')
if os.path.isdir(f'{SCALE}/pixel_linear'): EQ['linear_paper']=feats_dir(f'{SCALE}/pixel_linear',KS200,'eq_linear_paper')   # the paper's field-linear on the same 200 pairs (reference only)
if os.path.isdir(f'{SCALE}/ot'): EQ['ot_paper']=feats_dir(f'{SCALE}/ot',KS200,'eq_ot_paper')
FE1=dict(np.load(f'{BASE}/correspondence/feats/E1.npz')); ks_ge=[k for k in KS120 if f'A_morph|{k}|inc' in FE1]
GE={c:np.stack([FE1[f'{c}|{k}|inc'] for k in ks_ge]).astype(np.float64) for c in ['A_morph','P_morph','PA_morph','A_fixA','P_fixA','PA_fixA']}; print('Gaussian-primitive features',len(ks_ge),'pairs')


In [ ]:
# ============================================================
# 2. PAIRED JACKKNIFE ΔKID — leave one pair out, recompute KID(A)−KID(B), SE = sqrt((n−1)/n · Σ(θ_i − θ̄)²); 95% CI = Δ ± 1.96 SE
# ============================================================
REF=ref_inc.astype(np.float64); dd=REF.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3
KYY=float((kk(REF,REF).sum()-len(REF))/(len(REF)*(len(REF)-1)))
def kid_terms(X): kxx=kk(X,X); np.fill_diagonal(kxx,0.0); return kxx, kk(X,REF).mean(1)
def kid_from(kxx,r,mask): m=int(mask.sum()); return float(kxx[np.ix_(mask,mask)].sum()/(m*(m-1))+KYY-2*r[mask].mean())
def jack_dkid(XA,XB):
    n=len(XA); (kA,rA),(kB,rB)=kid_terms(XA),kid_terms(XB); full=np.ones(n,bool)
    d=kid_from(kA,rA,full)-kid_from(kB,rB,full); loo=np.empty(n)
    for i in range(n):
        m=full.copy(); m[i]=False; loo[i]=kid_from(kA,rA,m)-kid_from(kB,rB,m)
    se=float(np.sqrt((n-1)/n*((loo-loo.mean())**2).sum())); return d,d-1.96*se,d+1.96*se
def kid_point(X): k,r=kid_terms(X); return kid_from(k,r,np.ones(len(X),bool))
rows=[]
def add(name,XA,XB,label):
    d,lo,hi=jack_dkid(XA,XB); rows.append(dict(macro=name,comparison=label,dKID=d,lo=lo,hi=hi,excludes0=lo*hi>0,n=len(XA)))
# Texture Mixer coupling ablation (120 pairs): each rule vs the native positional lerp, and entropic vs mean
add('TMdKIDmean',TM['mean'],TM['lerp'],'TM target-mean − lerp'); add('TMdKIDot',TM['ot_r0.03'],TM['lerp'],'TM entropic − lerp'); add('TMdKIDhard',TM['hard'],TM['lerp'],'TM hard − lerp'); add('TMdKIDotVsMean',TM['ot_r0.03'],TM['mean'],'TM entropic − mean')
# equal-sample CNT ablation (200 pairs)
add('EqdKIDhard',EQ['hard'],EQ['ot_e0.05'],'hard − entropic 0.05'); add('EqdKIDhardSharp',EQ['hard'],EQ['ot_e0.005'],'hard − entropic 0.005'); add('EqdKIDsharp',EQ['ot_e0.005'],EQ['ot_e0.05'],'entropic 0.005 − 0.05')
if 'linear_paper' in EQ: add('EqdKIDhardVsLinear',EQ['hard'],EQ['linear_paper'],'hard − field-linear (paper render)'); add('EqdKIDotVsLinear',EQ['ot_e0.05'],EQ['linear_paper'],'entropic 0.05 − field-linear (paper render)')
# Gaussian-primitive study (120 pairs): moving layout, appearance vs position cost; frozen layout, appearance vs position
add('GEdKIDmorph',GE['A_morph'],GE['P_morph'],'primitives moving: appearance − position'); add('GEdKIDfix',GE['A_fixA'],GE['P_fixA'],'primitives frozen: appearance − position')
J=pd.DataFrame(rows); J.to_csv(f'{JK}/jackknife_dKID.csv',index=False)
print('point KIDs:',{**{f'TM {r}':round(kid_point(X),4) for r,X in TM.items()},**{f'EQ {r}':round(kid_point(X),4) for r,X in EQ.items()},**{f'GE {c}':round(kid_point(X),4) for c,X in GE.items()}})
print(); print(J.round(4).to_string(index=False)); print()
print('% paired leave-one-pair-out jackknife, 95% CI — replaces the m-of-n values in the numbers file')
for _,r in J.iterrows(): print(f'\\newcommand{{\\{r.macro}}}{{{r.dKID:+.4f}}}\\newcommand{{\\{r.macro}CI}}{{[{r.lo:+.4f},{r.hi:+.4f}]}}   % {r.comparison}: {"excludes 0" if r.excludes0 else "INCLUDES 0"}')


In [ ]:
# ============================================================
# ONE CONSISTENT SET — (1) field-linear re-rendered through the OT pipeline on the 200 pairs, (2) every 120-pair row rescored in
# this session against the canonical reference, (3) paired jackknife ΔKID on the final images. Prints the macro block.
# ============================================================
from scipy import linalg as _sl
SCALE=f'{BASE}/tm_scale1000'; JK=f'{BASE}/jackknife'; os.makedirs(JK,exist_ok=True); t0=time.time()
KS120=sorted(int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k); KS200=[int(p_['k']) for p_ in MAN_ALL[:200]]
REF=ref_inc.astype(np.float64); dd=REF.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3; KYY=float((kk(REF,REF).sum()-len(REF))/(len(REF)*(len(REF)-1)))
def kid_terms(X): kxx=kk(X,X); np.fill_diagonal(kxx,0.0); return kxx,kk(X,REF).mean(1)
def kid_from(kxx,r,m): n=int(m.sum()); return float(kxx[np.ix_(m,m)].sum()/(n*(n-1))+KYY-2*r[m].mean())
def kid_point(X): k,r=kid_terms(X); return kid_from(k,r,np.ones(len(X),bool))
def fid(X): mx,my=X.mean(0),REF.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(REF,rowvar=False); c=_sl.sqrtm(Sx@Sy); c=c.real if np.iscomplexobj(c) else c; return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*c))
def jack(XA,XB):
    n=len(XA); (kA,rA),(kB,rB)=kid_terms(XA),kid_terms(XB); full=np.ones(n,bool); d=kid_from(kA,rA,full)-kid_from(kB,rB,full); loo=np.empty(n)
    for i in range(n): m=full.copy(); m[i]=False; loo[i]=kid_from(kA,rA,m)-kid_from(kB,rB,m)
    se=float(np.sqrt((n-1)/n*((loo-loo.mean())**2).sum())); return d,d-1.96*se,d+1.96*se
# ---------- (1) field-linear through the OT pipeline, 200 pairs ----------
LIN=f'{OUT}/eqsample/linear_cp_t0.5'; os.makedirs(LIN,exist_ok=True)
for k in KS200:
    fp=f'{LIN}/{k:04d}.png'
    if os.path.exists(fp): continue
    d=get_d(k); L,Cc,H,W,sf,tf=_grids(d); to_pil(_dec(d,0.5*sf+0.5*tf,L,H,W,Cc)).save(fp)   # same field blend, same pyramid construction as the coupling rows
print(f'field-linear (consistent pyramid) rendered {time.time()-t0:.0f}s')
# ---------- score helper: canonical score_image + inception feats, cached per directory ----------
def score_dir(name,d,ks):
    fp=f'{JK}/scores_{name}.csv'; fq=f'{JK}/feats_{name}.npy'
    if os.path.exists(fp) and os.path.exists(fq): return pd.read_csv(fp),np.load(fq)
    rows,F=[],[]
    for k in ks:
        s,fi,fd=score_image(k,Image.open(f'{d}/{k:04d}.png').convert('RGB')); rows.append(dict(k=k,**s)); F.append(fi)
    S=pd.DataFrame(rows); F=np.stack(F).astype(np.float64); S.to_csv(fp,index=False); np.save(fq,F); return S,F
def summ(S,F): return dict(KID=kid_point(F),FID=fid(F),RInc=float(S.real_inc.median()),RDin=float(S.real_dino.median()),Bal=float((S.place-0.5).abs().median()),Place=float(S.place.median()))
# ---------- (2) the 120-pair table, one session ----------
def first_dir(cands): return next((p for p in cands if os.path.isdir(p)),None)
D120={'pix':f'{SCALE}/pixel_linear','tex':f'{SCALE}/texton_linear','gauss':f'{SCALE}/gaussian_w2','ot':f'{SCALE}/ot','hard':f'{SCALE}/exact','tm':f'{SCALE}/mid','gpt':f'{SCALE}/gpt',
      'gptsym':first_dir([f'{SCALE}/gpt_sym',f'{SCALE}/gpt_symmetric',f'{SCALE}/gpt_p1',f'{SCALE}/gpt2']),'tmlerp':f'{SCALE}/tmc/lerp','tmmean':f'{SCALE}/tmc/mean','tmot':f'{SCALE}/tmc/ot_r0.03','tmhard':f'{SCALE}/tmc/hard'}
D120={n:p for n,p in D120.items() if p and os.path.isdir(p)}; print('120-pair dirs found:',list(D120))
R120={n:score_dir(f'g_{n}',p,KS120) for n,p in D120.items()}; T120=pd.DataFrame({n:summ(S,F) for n,(S,F) in R120.items()}).T
print(f'\n=== 120 pairs, one session, canonical reference ({time.time()-t0:.0f}s) ==='); print(T120.round(4).to_string())
# ---------- (3) equal-sample rows incl. the consistent linear, and all jackknives ----------
D200={'linear_cp':LIN,'hard':f'{OUT}/eqsample/hard_t0.5','ot_e0.005':f'{OUT}/eqsample/ot_e0.005_t0.5','ot_e0.05':f'{OUT}/eqsample/ot_e0.05_t0.5','ot_paper':f'{SCALE}/ot','linear_paper':f'{SCALE}/pixel_linear'}
R200={n:score_dir(f'eq_{n}',p,KS200) for n,p in D200.items() if os.path.isdir(p)}; T200=pd.DataFrame({n:summ(S,F) for n,(S,F) in R200.items()}).T
print(f'\n=== 200 equal-sample pairs, one session ({time.time()-t0:.0f}s) ==='); print(T200.round(4).to_string())
F200={n:F for n,(S,F) in R200.items()}; F120={n:F for n,(S,F) in R120.items()}; J=[]
def add(name,XA,XB,lab): d,lo,hi=jack(XA,XB); J.append(dict(macro=name,comparison=lab,dKID=d,lo=lo,hi=hi,excl0=lo*hi>0,n=len(XA)))
add('EqdKIDotVsLinear',F200['ot_e0.05'],F200['linear_cp'],'entropic 0.05 (1024) − field-linear, same pipeline'); add('EqdKIDotPaperVsLinear',F200['ot_paper'],F200['linear_cp'],'entropic 0.05 (4096) − field-linear, same pipeline')
add('EqdKIDhardVsLinear',F200['hard'],F200['linear_cp'],'hard − field-linear, same pipeline'); add('EqdKIDhard',F200['hard'],F200['ot_e0.05'],'hard − entropic 0.05')
add('EqdKIDhardSharp',F200['hard'],F200['ot_e0.005'],'hard − entropic 0.005'); add('EqdKIDsharp',F200['ot_e0.005'],F200['ot_e0.05'],'entropic 0.005 − 0.05')
if 'tmlerp' in F120:
    for n,mac in [('tmmean','TMdKIDmean'),('tmot','TMdKIDot'),('tmhard','TMdKIDhard')]: add(mac,F120[n],F120['tmlerp'],f'TM {n} − lerp')
    add('TMdKIDotVsMean',F120['tmot'],F120['tmmean'],'TM entropic − mean')
if 'ot' in F120 and 'pix' in F120: add('GdKIDotVsPix',F120['ot'],F120['pix'],'120 pairs: AAT − field-linear')
J=pd.DataFrame(J); J.to_csv(f'{JK}/jackknife_final.csv',index=False); T120.to_csv(f'{JK}/table120_final.csv'); T200.to_csv(f'{JK}/table200_final.csv')
print('\n'+J.round(4).to_string(index=False))
# ---------- macro block ----------
print('\n% ===== ONE SESSION, CANONICAL REFERENCE — paste into the numbers file, replacing existing names =====')
NM={'pix':'pix','tex':'tex','gauss':'gauss','ot':'ot','hard':'hard','tm':'tm','gpt':'gpt','gptsym':'gptsym'}
for n,s in NM.items():
    if n in T120.index: r=T120.loc[n]; print(f'\\newcommand{{\\GFID{s}}}{{{r.FID:.1f}}}\\newcommand{{\\GKID{s}}}{{{r.KID:.4f}}}\\newcommand{{\\GRInc{s}}}{{{r.RInc:.3f}}}\\newcommand{{\\GRDin{s}}}{{{r.RDin:.3f}}}\\newcommand{{\\GPlace{s}}}{{{r.Bal:.3f}}}')
for n,s in [('tmlerp','lerp'),('tmmean','mean'),('tmot','ot'),('tmhard','hard')]:
    if n in T120.index: r=T120.loc[n]; print(f'\\newcommand{{\\TMKID{s}}}{{{r.KID:.4f}}}\\newcommand{{\\TMFID{s}}}{{{r.FID:.1f}}}\\newcommand{{\\TMRInc{s}}}{{{r.RInc:.3f}}}\\newcommand{{\\TMRDin{s}}}{{{r.RDin:.3f}}}\\newcommand{{\\TMBal{s}}}{{{r.Bal:.3f}}}')
for n,s in [('linear_cp','linear'),('hard','hard'),('ot_e0.005','otSharp'),('ot_e0.05','ot'),('ot_paper','paper')]:
    if n in T200.index: r=T200.loc[n]; print(f'\\newcommand{{\\EqKID{s}}}{{{r.KID:.4f}}}\\newcommand{{\\EqFID{s}}}{{{r.FID:.1f}}}\\newcommand{{\\EqBal{s}}}{{{r.Bal:.3f}}}\\newcommand{{\\EqRInc{s}}}{{{r.RInc:.3f}}}')
g=100*(T200.loc['linear_cp','KID']-T200.loc['ot_paper','KID'])/T200.loc['linear_cp','KID']; gf=100*(T200.loc['linear_cp','FID']-T200.loc['ot_paper','FID'])/T200.loc['linear_cp','FID']
print(f'\\newcommand{{\\EqKIDgainOT}}{{{g:.0f}\\%}}\\newcommand{{\\EqFIDgainOT}}{{{gf:.0f}\\%}}   % 4096-sample operator vs field-linear, same pipeline')
for _,r in J.iterrows(): print(f'\\newcommand{{\\{r.macro}}}{{{r.dKID:+.4f}}}\\newcommand{{\\{r.macro}CI}}{{[{r.lo:+.4f},{r.hi:+.4f}]}}   % {r.comparison}: {"excludes 0" if r.excl0 else "INCLUDES 0"}')

In [ ]:
# ============================================================
# KID FIX — subtract the reference kernel's trace, not n; recompute every absolute KID and percentage from cached features; ΔKIDs unchanged
# ============================================================
REF=ref_inc.astype(np.float64); dd=REF.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3
K_ref=kk(REF,REF); KYY=float((K_ref.sum()-np.trace(K_ref))/(len(REF)*(len(REF)-1))); del K_ref
def kid(X): kxx=kk(X,X); np.fill_diagonal(kxx,0.0); m=len(X); return float(kxx.sum()/(m*(m-1))+KYY-2*kk(X,REF).mean())
JK=f'{BASE}/jackknife'; L=lambda n: np.load(f'{JK}/feats_{n}.npy').astype(np.float64)
G={n:kid(L(f'g_{n}')) for n in ['pix','tex','gauss','ot','hard','tm','gpt','gptsym','tmlerp','tmmean','tmot','tmhard']}
E={n:kid(L(f'eq_{n}')) for n in ['linear_cp','hard','ot_e0.005','ot_e0.05','ot_paper']}
FE1=dict(np.load(f'{BASE}/correspondence/feats/E1.npz')); ks=sorted({int(k.split('|')[1]) for k in FE1 if k.startswith('A_morph|') and k.endswith('|inc')})
GE={c:kid(np.stack([FE1[f'{c}|{k}|inc'] for k in ks]).astype(np.float64)) for c in ['P_fixA','A_fixA','PA_fixA','P_morph','A_morph','PA_morph']}
FE4=dict(np.load(f'{BASE}/correspondence/feats/E4.npz')); FED=dict(np.load(f'{BASE}/correspondence/feats/dense.npz')); EM=json.load(open(f'{BASE}/correspondence/E4_match.json'))['eps_match']
ks4=[k for k in ks if f'gA_sink{EM}_fixA|{k}|inc' in FE4 and f'ot|{k}|inc' in FED]
prim_soft=kid(np.stack([FE4[f'gA_sink{EM}_fixA|{k}|inc'] for k in ks4])); dense_ot=kid(np.stack([FED[f'ot|{k}|inc'] for k in ks4]))
gK=100*(E['linear_cp']-E['ot_paper'])/E['linear_cp']; gK1=100*(E['linear_cp']-E['ot_e0.05'])/E['linear_cp']
print('corrected absolute KIDs (ΔKIDs and CIs unchanged):'); print({**{f'G {n}':round(v,4) for n,v in G.items()},**{f'Eq {n}':round(v,4) for n,v in E.items()},**{f'GE {c}':round(v,4) for c,v in GE.items()},'prim_soft':round(prim_soft,4),'dense_ot':round(dense_ot,4)})
print('\n% ===== corrected KID macros (trace fix) =====')
for n,s in [('pix','pix'),('tex','tex'),('gauss','gauss'),('ot','ot'),('hard','hard'),('tm','tm'),('gpt','gpt'),('gptsym','gptsym')]: print(f'\\newcommand{{\\GKID{s}}}{{{G[n]:.4f}}}')
for n,s in [('tmlerp','lerp'),('tmmean','mean'),('tmot','ot'),('tmhard','hard')]: print(f'\\newcommand{{\\TMKID{s}}}{{{G[n]:.4f}}}')
for n,s in [('linear_cp','linear'),('hard','hard'),('ot_e0.005','otSharp'),('ot_e0.05','ot'),('ot_paper','paper')]: print(f'\\newcommand{{\\EqKID{s}}}{{{E[n]:.4f}}}')
print(f'\\newcommand{{\\EqKIDgainOT}}{{{gK:.0f}\\%}}\\newcommand{{\\EqKIDgainOTsmall}}{{{gK1:.0f}\\%}}')
print(f'\\newcommand{{\\GEKIDfixP}}{{{GE["P_fixA"]:.3f}}}\\newcommand{{\\GEKIDfixA}}{{{GE["A_fixA"]:.3f}}}\\newcommand{{\\GEKIDfixPA}}{{{GE["PA_fixA"]:.3f}}}\\newcommand{{\\GEKIDmorphP}}{{{GE["P_morph"]:.3f}}}\\newcommand{{\\GEKIDmorphA}}{{{GE["A_morph"]:.3f}}}')
print(f'\\newcommand{{\\GEKIDprimSoft}}{{{prim_soft:.3f}}}\\newcommand{{\\GEKIDdenseOT}}{{{dense_ot:.3f}}}')
print(f'\\newcommand{{\\TMKIDgainMean}}{{{100*(G["tmlerp"]-G["tmmean"])/G["tmlerp"]:.0f}\\%}}\\newcommand{{\\TMKIDgainOT}}{{{100*(G["tmlerp"]-G["tmot"])/G["tmlerp"]:.0f}\\%}}')